# MimicKit 一键预览（真实款机器人）

用 [xbpeng/MimicKit](https://github.com/xbpeng/MimicKit) 的 **预训练 ckpt** + **Isaac Lab v2.3+** 后端，5 个动作 + 1 个 mocap 回放，全部一键复现。

| 动作 | 机器人 | 算法 | 物理仿真 |
|---|---|---|---|
| `spinkick` | Unitree G1 | DeepMimic PPO | ✅ |
| `jump` (double-kong) | Unitree G1 | DeepMimic PPO | ✅ |
| `run` | Unitree G1 | ADD | ✅ |
| `walk` | Unitree G1 | LCP | ✅ |
| `cartwheel` | Unitree G1 | — (无 ckpt) | ❌ mocap kinematic 回放 |
| `go2_pace` | Unitree Go2 (四足) | DeepMimic PPO | ✅ |

蓝色 = policy 控制的物理本体；绿色 = mocap 参考影子（`ref_char_offset` 偏 2m 显示）。

---

## 0) 一次性准备（首次运行才需要）

### a. 拉 MimicKit submodule

In [ ]:
!git submodule update --init --recursive dependencies/MimicKit

### b. 下载 + 解压 MimicKit 资产包（assets/models/motions，~384 MB）

**SharePoint 链接强制走 session cookie，curl 拿不到 → 浏览器手动下载：**

1. 打开 [MimicKit_Data.zip @ SharePoint](https://1sfu-my.sharepoint.com/:u:/g/personal/xbpeng_sfu_ca/EclKq9pwdOBAl-17SogfMW0Bved4sodZBQ_5eZCiz9O--w?e=bqXBaa)
2. 下载到 `~/下载/MimicKit_Data.zip`（或调整下面 cell 里的 `ZIP_PATH`）
3. 跑下面 cell 解压合并到 `dependencies/MimicKit/data/`（已有同名文件不覆盖，submodule 干净）

In [ ]:
ZIP_PATH = '~/下载/MimicKit_Data.zip'  # ← 改成你的实际路径
!test -f $ZIP_PATH && echo "✓ found zip" || echo "❌ zip not found at $ZIP_PATH"
!mkdir -p /tmp/mimickit_extract && unzip -q -o $ZIP_PATH -d /tmp/mimickit_extract/
!rsync -a --ignore-existing /tmp/mimickit_extract/MimicKit_Data/ dependencies/MimicKit/data/
!rm -rf /tmp/mimickit_extract
!ls -lh dependencies/MimicKit/data/models/ | head -8

### c. （可选）确认 conda env

脚本默认用 `CONDA_ENV=isaaclab`，要换其他 env 在每次运行前 `export CONDA_ENV=xxx`。

In [ ]:
!conda env list | grep isaaclab || echo '⚠ isaaclab env not found — install Isaac Lab first'

---
## 1) G1 旋踢 spinkick （真物理 policy）

In [ ]:
!bash scripts/mimickit_preview.sh spinkick

## 2) G1 跳跃 jump（double-kong parkour，真物理 policy）

In [ ]:
!bash scripts/mimickit_preview.sh jump

## 3) G1 跑步 run （ADD 算法）

In [ ]:
!bash scripts/mimickit_preview.sh run

## 4) G1 走路 walk（LCP 算法）

In [ ]:
!bash scripts/mimickit_preview.sh walk

## 5) G1 翻车轮 cartwheel（mocap kinematic 回放，无物理）

MimicKit 没发 cartwheel 的 G1 ckpt，所以这里走 `view_motion` env：纯 mocap 关键帧贴到 G1 骨架上，看动作 silhouette。

In [ ]:
!bash scripts/mimickit_preview.sh cartwheel

## 6) Go2 四足 pace（真物理 policy）

In [ ]:
!bash scripts/mimickit_preview.sh go2_pace

---
## 7) LAFAN1 mocap 回放（无 ckpt → kinematic 预览，挑训练目标用）

从 [ember-lab-berkeley/LAFAN-G1](https://huggingface.co/datasets/ember-lab-berkeley/LAFAN-G1) 拿 Ubisoft LAFAN1 retargeted 到 G1 的数据，每段 ~2–4 分钟，专门用于挑训练目标。
运行前先转格式：


In [ ]:
# 1) 下载（首次）：openhe + ember-lab
!cd dependencies/MimicKit && python -c "from huggingface_hub import snapshot_download; \
snapshot_download('openhe/g1-retargeted-motions', repo_type='dataset', local_dir='data/motions/g1_extra/openhe', \
  allow_patterns=['kungfu_retargeted/*','ACCAD_retargeted/*','dance_db_retargeted/*','README.md']); \
snapshot_download('ember-lab-berkeley/LAFAN-G1', repo_type='dataset', local_dir='data/motions/g1_extra/ember_lab', \
  allow_patterns=['LAFAN_dance1_subject1_*','LAFAN_fight1_subject2_*','LAFAN_jumps1_subject1_*','LAFAN_run1_subject2_*'])"

# 2) 把 ember-lab .npz 转 MimicKit .pkl
import subprocess
for clip in ['dance1_subject1','fight1_subject2','jumps1_subject1','run1_subject2']:
    cat = clip.split('_')[0]  # dance1/fight1/jumps1/run1
    subprocess.run(['python','scripts/lafan_g1_npz_to_mimickit.py',
        '--input', f'dependencies/MimicKit/data/motions/g1_extra/ember_lab/LAFAN_{clip}_0_-1.npz',
        '--output', f'dependencies/MimicKit/data/motions/g1/lafan_{cat}.pkl'], check=True)


### 7a) LAFAN 格斗 (4 min)

In [ ]:
!bash scripts/mimickit_preview.sh lafan_fight

### 7b) LAFAN 跳跃 (4 min)

In [ ]:
!bash scripts/mimickit_preview.sh lafan_jumps

### 7c) LAFAN 舞蹈 (2 min)

In [ ]:
!bash scripts/mimickit_preview.sh lafan_dance

### 7d) LAFAN 奔跑 (4 min)

In [ ]:
!bash scripts/mimickit_preview.sh lafan_run

### 7e) LAFAN 格斗·5s 训练切段（warm-up，frames 0–150）
训练 pipeline 信心确认用。切段命令：`python scripts/lafan_g1_npz_to_mimickit.py --input <npz> --output <pkl> --start_frame 0 --end_frame 150`

In [ ]:
!bash scripts/mimickit_preview.sh lafan_fight_5s

### 7f) LAFAN 格斗·15s 训练切段（main，frames 600–1050，中段连续 punch / kick）
主训练目标。切段命令：`... --start_frame 600 --end_frame 1050`

In [ ]:
!bash scripts/mimickit_preview.sh lafan_fight_15s

---
## 7g) AMP 候选：8 个 dance 老师一键预览（挑 AMP 训练源）

LAFAN 的 dance 有 **2 套不同编舞 × 多 subject = 8 个连续 clip**（无 phrase 预分段）。
下面先把 8 个都拉下来做 **kinematic 回放预览**（纯 mocap 贴到 G1，无 policy），
你挑哪个/哪几个去训 **AMP**（AMP 吃整段、无 15s 长度限制，详见 `doc/mimickit_to_vla_dataset.html`）。

| 编舞 | subject | 时长 | clip |
|---|---|---|---|
| dance1 | 1/2/3 | **131.5 s** 每个 | `lafan_dance1_s{1,2,3}` |
| dance2 | 1/2/3/4/5 | **225.7 s** 每个 | `lafan_dance2_s{1..5}` |

> dance1 与 dance2 是**不同编舞**；同编舞不同 subject 是不同演员演同一套。选 AMP 源优先看编舞差异。


In [ ]:
# 7g.0) 下载 8 个 dance npz + 转 MimicKit pkl（full clip，首次 ~50MB）
from huggingface_hub import snapshot_download
from pathlib import Path
import subprocess, sys

REPO  = Path.cwd()
EMBER = REPO/'dependencies/MimicKit/data/motions/g1_extra/ember_lab'
MOTI  = REPO/'dependencies/MimicKit/data/motions/g1'
EMBER.mkdir(parents=True, exist_ok=True)

snapshot_download('ember-lab-berkeley/LAFAN-G1', repo_type='dataset',
                  local_dir=str(EMBER), allow_patterns='LAFAN_dance*.npz')

clips = [('dance1',s) for s in (1,2,3)] + [('dance2',s) for s in (1,2,3,4,5)]
for chor, subj in clips:
    npz = EMBER/f'LAFAN_{chor}_subject{subj}_0_-1.npz'
    out = MOTI/f'lafan_{chor}_s{subj}.pkl'
    if out.exists():
        print(f'✓ exists {out.name}'); continue
    subprocess.run([sys.executable, 'scripts/lafan_g1_npz_to_mimickit.py',
                    '--input', str(npz), '--output', str(out)], check=True)
print('✓ 8 dance teachers ready — 跑下面任意 cell 预览，关窗口看下一个')


**dance1 家族（同编舞，3 个演员）**


In [ ]:
!bash scripts/mimickit_preview.sh view lafan_dance1_s1


In [ ]:
!bash scripts/mimickit_preview.sh view lafan_dance1_s2


In [ ]:
!bash scripts/mimickit_preview.sh view lafan_dance1_s3


**dance2 家族（另一套编舞，5 个演员）**


In [ ]:
!bash scripts/mimickit_preview.sh view lafan_dance2_s1


In [ ]:
!bash scripts/mimickit_preview.sh view lafan_dance2_s2


In [ ]:
!bash scripts/mimickit_preview.sh view lafan_dance2_s3


In [ ]:
!bash scripts/mimickit_preview.sh view lafan_dance2_s4


In [ ]:
!bash scripts/mimickit_preview.sh view lafan_dance2_s5


---
## 8) 用我们自训的 LAFAN ckpt 启动 GUI eval

没有自己训练？无所谓——`mimickit_eval_chain.sh` 本地 `output/` 找不到 ckpt 时会自动 `snapshot_download` 从 [`wsagi/MimicKit-G1-LAFAN`](https://huggingface.co/wsagi/MimicKit-G1-LAFAN) 拉权重到 HF cache（`~/.cache/huggingface/`，内容寻址、零拷贝、commit 锁版本），并自动挂上 `g1_textured.usd` 还原 Unitree G1 原色。

和上面预览章节一样，每个 motion 就是一行 `!bash`：

| Motion | 质量 | 命令参数 |
|---|---|---|
| fight | 99 % 🟢 | `lafan_fight_15s` |
| dance | 98 % 🟢 | `lafan_dance_15s` |
| jumps | 98 % 🟢 | `lafan_jumps_15s` |
| run | 63 % 🟡 | `lafan_run_15s` |


### 8a) fight 15s（99 % 触顶，连续 punch / kick）


In [ ]:
!bash scripts/mimickit_eval_chain.sh lafan_fight_15s


### 8b) dance 15s（98 % 触顶）


In [ ]:
!bash scripts/mimickit_eval_chain.sh lafan_dance_15s


### 8b+) dance 30s（longer-horizon，warm-start 自 15s）

把 dance 时长翻倍到 **30s（900 帧 @30fps，`dance1_subject2`）**：从 15s dance ckpt 热启动跑 2500 iter，收敛 `Test_Return 244 > 15s 基线 227`（γ 折扣 return 饱和在同档天花板，追平/超过 = 全程覆盖住了）。

> 同一 clip 上 **AMP 拿不下**（判别器卡 0.98、跟不上节奏），DeepMimic phase-tracking 拿下了 —— dance 保真用 phase-tracking 不用 AMP。

没自训也能跑：本地无 ckpt 时自动从 [`wsagi/MimicKit-G1-LAFAN`](https://huggingface.co/wsagi/MimicKit-G1-LAFAN) 拉 `dance_30s/model.pt`。

In [ ]:
!bash scripts/mimickit_eval_chain.sh lafan_dance_30s


### 8c) jumps 15s（98 % 触顶）


In [ ]:
!bash scripts/mimickit_eval_chain.sh lafan_jumps_15s


### 8d) run 15s（63 % plateau，作为 baseline 留存）


In [ ]:
!bash scripts/mimickit_eval_chain.sh lafan_run_15s


### 8e) 4 个串行连看（关一个窗口进下一个）
不传参 = 默认顺序 fight → run → dance → jumps。


In [ ]:
!bash scripts/mimickit_eval_chain.sh


---
## 9) openhe/g1-retargeted-motions（仅参考，DoF 不匹配）

openhe 数据集用 [Mink retargeter](https://github.com/kevinzakka/mink) 把 SMPL → G1 23-DoF（去掉手腕 6 DoF），**和 MimicKit g1.xml 的 29-DoF 不兼容**，直接灌进 view_motion 会维度爆。

目录留在 `dependencies/MimicKit/data/motions/g1_extra/openhe/` 作为参考（ACCAD 113 clip / DanceDB 13 clip / kungfu 8 clip 含 Bruce_Lee_pose / Roundhouse_kick / Side_kick / Horse-stance_punch 等）。要用上得：

1. 给 MimicKit 加一份 g1_23dof.xml 模型（剥手腕 actuator）
2. 或者写 23→29 padding（手腕填 0），物理上不严谨但视觉够用

暂不在 driver 里启用。

---
## 调试备忘

- **viewer 不显示**：检查 `DISPLAY` 环境变量；在 SSH 远端要 `ssh -X` 或 VNC/RDP
- **第一次启动慢**：Isaac Sim 5.1 kit 加载 + USD shader 编译，首次 30–60s
- **API drift 报错** `traverse_instance_prims`：脚本已自动 `git apply patches/mimickit/isaaclab-v23-api.patch`；如果再坏可手动 reset：`cd dependencies/MimicKit && git checkout mimickit/engines/isaac_lab_engine.py && cd - && bash scripts/mimickit_preview.sh spinkick`
- **想训练而非 inference**：把 `--mode test` 改 `--mode train`，去掉 `--model_file`，加 `--num_envs 4096`，参考 `dependencies/MimicKit/docs/README_DeepMimic.md`
- **想换其他 ckpt**：`ls dependencies/MimicKit/data/models/` 看全集，照 `scripts/mimickit_preview.sh` 的 case 表加新 profile

### 相关 memory
- `.memory/` 暂无 MimicKit 专项；以后踩到坑可以加 `mimickit-isaaclab-v23-patch.md`